In [2]:
# Nếu chưa cài đặt
# Import thư viện
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ultralytics import YOLO
from sklearn.metrics import classification_report, confusion_matrix
from IPython.display import display, Image, HTML


In [5]:
# Nhãn cử chỉ
class_names = ['Hello', 'IloveYou', 'No', 'Please', 'Thanks', 'Yes']

# Đường dẫn ảnh test
test_dir = './Data/test/images'  # Thay nếu thư mục khác

# Load mô hình YOLO
model = YOLO('best.pt')  # Thay đường dẫn mô hình nếu khác

# Tạo thư mục lưu ảnh định tính
correct_dir = './output/correct'
incorrect_dir = './output/incorrect'
for cname in class_names:
    os.makedirs(os.path.join(correct_dir, cname), exist_ok=True)
    os.makedirs(os.path.join(incorrect_dir, cname), exist_ok=True)


In [6]:
y_true = []
y_pred = []

for fname in os.listdir(test_dir):
    if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
        continue

    true_label = next((c for c in class_names if fname.lower().startswith(c.lower())), None)
    if true_label is None:
        continue

    img_path = os.path.join(test_dir, fname)
    img = cv2.imread(img_path)
    if img is None:
        continue

    results = model.predict(source=img, conf=0.25, verbose=False)
    pred_label = (
        results[0].names[int(results[0].boxes.cls[0])]
        if results[0].boxes else "No Detection"
    )

    y_true.append(true_label)
    y_pred.append(pred_label)

    annotated_img = results[0].plot()
    save_path = os.path.join(
        correct_dir if pred_label == true_label else incorrect_dir,
        true_label,
        fname
    )
    cv2.imwrite(save_path, annotated_img)


In [7]:
def show_images(folder, label, count=3):
    folder_path = os.path.join(folder, label)
    if not os.path.exists(folder_path):
        print(f'Không tìm thấy thư mục {folder_path}')
        return

    files = os.listdir(folder_path)[:count]
    images_html = ''
    for f in files:
        image_path = os.path.join(folder_path, f)
        images_html += f'<img src="files/{image_path}" width="200" style="margin: 5px;">'
    display(HTML(images_html))


In [8]:
for gesture in class_names:
    print(f'✅ Ví dụ đúng: {gesture}')
    show_images(correct_dir, gesture)
    
    print(f'❌ Ví dụ sai: {gesture}')
    show_images(incorrect_dir, gesture)


✅ Ví dụ đúng: Hello


❌ Ví dụ sai: Hello


✅ Ví dụ đúng: IloveYou


❌ Ví dụ sai: IloveYou


✅ Ví dụ đúng: No


...

❌ Ví dụ sai: Please


✅ Ví dụ đúng: Thanks


❌ Ví dụ sai: Thanks


✅ Ví dụ đúng: Yes


❌ Ví dụ sai: Yes
